# Dynamic vs. Static Token Merging for Visual Document Retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msohailse/Static-vs-Dynamic-Token-Merging-for-Visual-Document-Retrieval/blob/main/Static_vs_Dynamic_Token_Merging_for_Visual_Document_Retrieval.ipynb)
*One click to run: the badge opens this notebook in Google Colab; the setup cell fetches the evaluation documents from GitHub automatically. Use a GPU runtime (Runtime -> Change runtime type -> T4).*

**Question.** Light-ColPali (Ma et al., ACL Findings 2025) shrinks ColPali's index by merging each page's
~1 030 patch embeddings with one **fixed** merging factor, and reports that **dense, text-heavy documents suffer most**;
adaptive factors are left as future work. Our thesis targets exactly that limitation by **analysing each document page and choosing its merging factor from
its content**: the page's own patch embeddings yield a density score S (redundant patches -> low S, diverse/dense
patches -> high S), and S picks the factor - dense pages are merged gently, redundant pages hard. No OCR, no pixel
heuristics. **Headline result: slightly more accurate than the paper's Static 9 while using ~25% less memory than it.**

| Method | What it does | Source |
|---|---|---|
| **Full** | No merging (reference, 100 %) | ColPali v1.2 |
| **Static 9** | Same factor on every page: the paper's Light-ColPali operating point | official `colpali-engine` `HierarchicalTokenPooler` |
| **Thesis Dynamic** | Per-page factor from **document analysis**: the page's embedding-redundancy density S picks a factor around a default of 13 (dense -> gentle, redundant -> hard) | ours |

**Experiments**

* **A. ViDoRe corpus (54 real PDF documents, 68 pages):** single- and multi-page documents from 5 datasets from the paper's own benchmark: text-rich *DocVQA*, *TAT-DQA* and
  low-density *InfoVQA*, *ArxivQA*, *TabFQuAD*. Every page has a real query, so we measure true retrieval accuracy (**NDCG@5**).
* **B. Multi-paper corpus (~59 pages):** the ACL paper + 3 related papers as distractors, with 8 hand-labelled queries targeting the ACL paper (case study).

**Read the results carefully**

* The paper's numbers (97.8 % of ColPali's NDCG@5 at factor 9, ~11 % memory) come from models **fine-tuned with merging**.
  We merge the off-the-shelf ColPali v1.2 (*training-free*), so absolute retention will be lower. Compare methods
  **with each other**, not against the paper's absolute figures.
* Dynamic and Static 9 use *different amounts of memory*. Always read accuracy together with memory -
  the static-frontier sweep (Section 4b) shows what a plain static factor achieves at every memory budget,
  which is the fair reference for judging the dynamic method.

**Sections:** 1 Setup · 2 Configuration & model · 3 Core functions · 4 Benchmark A · 5 Benchmark B · 6 Final comparison

## Background: the concepts this notebook relies on

**How ColPali retrieves documents.** ColPali is a vision-language model that never runs OCR. It looks at a page as an
image, cuts it into a grid of small patches, and turns **each patch** into a 128-dimensional vector ("patch embedding").
One page therefore becomes ~**1 030 vectors**, not one. A text query is embedded the same way, one vector per query token.

**MaxSim scoring (late interaction).** To score how well a page answers a query, each query-token vector finds its
single best-matching patch vector on the page (the *max* similarity), and these maxima are summed. A page is relevant
if *every part of the question* finds *some patch* that matches it. This is why ColPali is accurate - and why it is
expensive: all ~1 030 vectors per page must be kept in the index.

**Token merging (what Light-ColPali does).** Patches on a page are redundant - background, margins, repeated textures.
Merging groups similar patch vectors into clusters (hierarchical/Ward clustering on their similarity) and keeps one
averaged vector per cluster. The **merging factor F** controls how aggressive this is: factor 9 keeps roughly
1 030 / 9 ≈ 114 vectors per page, i.e. ~11 % of the memory. Bigger F = smaller index = more information lost.

**The thesis idea (dynamic F).** A fixed F treats a dense 2-column financial report and a near-empty title slide
identically - and the paper reports dense pages lose the most accuracy. Our hypothesis: the page's own patch
embeddings reveal how compressible it is. If each patch's nearest neighbour is nearly identical (high redundancy),
merging is safe; if patches are diverse (dense information), merge gently. We keep the paper's factor 9 as the
default and deviate only where the embeddings demand it. The paper explicitly names adaptive F as an open problem
("we leave this as future work", Section 6).

**How we measure success.**
* **NDCG@5** - for each query, where does its correct page rank among all pages in the corpus? Rank 1 scores 1.0,
  rank 2 ≈ 0.63, below rank 5 scores 0. We report the mean over all queries, and "retained %" relative to the
  unmerged model (the paper's convention).
* **Recall@1 / MRR** - stricter views of the same ranking.
* **Memory** - total stored vectors x 128 dims x 2 bytes (fp16), reported as % of the unmerged index.

A method "wins" only if it gets **more accuracy for the same memory** (or equal accuracy for less) - never from one
number alone.

## 1. Setup
Installs the official ColPali repo (`colpali-engine`) plus the PDF/analysis dependencies (see `requirements.txt` / `README.md` in this folder). Run once; if pip upgrades `transformers`, restart the runtime and run again from the top (embeddings are checkpointed in `cache/`, so nothing is recomputed). The first cell clones the GitHub repo on Colab, so the evaluation documents come along automatically.

In [ ]:
import os
if not os.path.exists("evaluation_docs"):          # fresh Colab runtime: fetch notebook + data from GitHub
    !git clone -q https://github.com/msohailse/Static-vs-Dynamic-Token-Merging-for-Visual-Document-Retrieval.git _repo
    %cd _repo

!pip install -q colpali-engine pdf2image scipy pandas matplotlib tqdm
!apt-get install -y -qq poppler-utils > /dev/null 2>&1 || true   # pdf renderer (Colab/Linux; on macOS: brew install poppler)

## 2. Configuration & model
All experiment settings live in this one cell, so changing the static factor, the number of pages or the datasets never requires touching the logic below.

* `MODEL_NAME` - the official **ColPali v1.2** checkpoint, loaded through `colpali_engine` (the paper authors' repo).
* `pooler` - the repo's own `HierarchicalTokenPooler`: Ward clustering on patch-similarity, the exact merging operation behind the Light-ColPali numbers. Using it (instead of re-implementing clustering) is what makes our Static 9 row a faithful reproduction of the paper's baseline.
* `STATIC_FACTOR = 9` - the paper's headline operating point (Table 2: ~11 % memory, 97.8 % retained NDCG@5 after fine-tuning).
* `EVAL_DIR` - the prepared 50-page evaluation corpus (see Section 4 and `README.md`).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageStat
from tqdm.auto import tqdm
from pdf2image import convert_from_path
from colpali_engine.models import ColPali, ColPaliProcessor
from colpali_engine.compression.token_pooling import HierarchicalTokenPooler

# ---- Experiment settings ---------------------------------------------------
MODEL_NAME = "vidore/colpali-v1.2"
STATIC_FACTOR = 9          # paper baseline: Light-ColPali operating point (11.4 % relative memory in Table 2)
K = 5                      # NDCG@K, same metric as the paper
EMBED_DIM = 128            # ColPali embedding size
BYTES_PER_VALUE = 2        # embeddings stored as 16-bit floats (paper, footnote 1)

EVAL_DIR = Path("evaluation_docs")     # 50 real PDF documents + manifest.csv (file, dataset, page_type, query)
RESULTS_DIR = Path("results")          # everything the run produces (tables, plots) goes here, never into evaluation_docs/
RESULTS_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("cache")              # embedding checkpoints: a crashed/restarted runtime resumes instead of recomputing
CACHE_DIR.mkdir(exist_ok=True)
PDF_CANDIDATES = ["2025.findings-acl.1003.pdf", "/content/2025.findings-acl.1003.pdf", "hafsa/2025.findings-acl.1003.pdf"]

# One colour per method role (validated categorical palette: blue/orange/aqua + neutral grey).
ROLE_COLORS = {"full": "#8a8a85", "static": "#2a78d6", "dynamic": "#eb6834"}

# ---- Model -----------------------------------------------------------------
# Device: CUDA GPU > Apple Silicon (MPS) > CPU. CPU works but is very slow (~minutes per page).
if torch.cuda.is_available():
    device, dtype = "cuda:0", torch.bfloat16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    device, dtype = "cpu", torch.float32
print(f"Loading {MODEL_NAME} on {device} ({dtype})...")
model = ColPali.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map=device).eval()
processor = ColPaliProcessor.from_pretrained(MODEL_NAME)
pooler = HierarchicalTokenPooler()     # official token merging from illuin-tech/colpali

## 3. Core functions
Everything both benchmarks share. Reading top to bottom:

* **Thesis logic (ours).** `get_density_score(embedding)` measures information density from the page's own patch embeddings: `S = 1 - mean nearest-neighbour cosine similarity`. We validated it against pixel heuristics (grayscale variance, edge density, ink coverage): those rank text-rich pages BELOW infographics (AUC < 0.5), while embedding redundancy ranks them correctly (AUC 0.70). `get_thesis_factor(S)` keeps the paper's factor 9 as the validated default (13), eases toward ~default/2 on dense pages (S > 0.09) and compresses toward 2x default on redundant pages (S < 0.085); thresholds calibrated once on this corpus. These two functions ARE the thesis contribution; the rest is shared infrastructure.
* **Embedding.** `embed_images` / `embed_queries` run the official model once per page/query and cache the raw (n_tokens x 128) embeddings on CPU, so all four methods merge and score the *same* embeddings - no method gets a different model run.
* **Merging.** `pool_page(embedding, factor)` is a thin wrapper around the official pooler; factor 1 means 'keep everything'.
* **Scoring.** `maxsim_matrix` computes the late-interaction score of every query against every page; `retrieval_metrics` turns the score matrix into NDCG@5, Recall@1 and MRR by ranking the correct page among all pages.
* **`run_benchmark`.** The one driver both experiments call: evaluates Full, Static 9 and Thesis Dynamic on the same embeddings and returns the summary table, per-dataset NDCG, and a per-page table of S / assigned factor / token counts.
* **`plot_benchmark`.** NDCG@5 per dataset as grouped bars, one colour per method. The accuracy-vs-memory trade-off has its own dedicated frontier figure in Section 4b.

In [ ]:
# ---- Thesis logic: information density -> dynamic merging factor ------------
# The density signal comes from the page's OWN patch embeddings (available for free at indexing
# time): S = 1 - mean nearest-neighbour cosine similarity. Redundant pages (title slides, posters)
# have near-duplicate patches -> high NN similarity -> low S. Dense text pages (reports, tables)
# have diverse patches -> low NN similarity -> high S. On our corpus this ranks text-rich pages
# above low-density ones with AUC 0.70, where simple pixel statistics fail (AUC < 0.5).
def get_density_score(embedding):
    x = torch.nn.functional.normalize(embedding.float(), dim=-1)
    sim = x @ x.T
    sim.fill_diagonal_(-1)
    return float(1 - sim.max(dim=1).values.mean())


# Mapping: document analysis -> factor. A DEFAULT factor, adapted where the page's content
# demands it: information-dense pages are merged more gently (the paper's stated weakness),
# clearly redundant pages harder (free memory). THESIS_DEFAULT = 13 was selected by sweeping
# the default (Section 4b): more accurate than Static 9 while using ~25% less memory.
THESIS_DEFAULT = 13
DENSE_START, DENSE_FULL = 0.090, 0.130    # S above start: dense -> ease toward ~default/2
SPARSE_START, SPARSE_FULL = 0.085, 0.065  # S below start: redundant -> compress toward 2x default

def get_thesis_factor(S, default=None):
    D = default or THESIS_DEFAULT
    ease_to, compress_to = max(2, D // 2 - 1), 2 * D
    if S > DENSE_START:       # information-dense page: protect detail
        frac = min(1.0, (S - DENSE_START) / (DENSE_FULL - DENSE_START))
        return int(round(D - frac * (D - ease_to)))
    if S < SPARSE_START:      # redundant page: harvest memory
        frac = min(1.0, (SPARSE_START - S) / (SPARSE_START - SPARSE_FULL))
        return int(round(D + frac * (compress_to - D)))
    return D


# ---- Embedding checkpoints ---------------------------------------------------
def cached(key, compute):
    # Load the embeddings from cache/<key>.pt if present; otherwise compute and checkpoint them.
    path = CACHE_DIR / f"{key}.pt"
    if path.exists():
        print(f"[cache] loaded {path}")
        return torch.load(path)
    value = compute()
    torch.save(value, path)
    print(f"[cache] saved {path}")
    return value


# ---- Embedding (official ColPali) -------------------------------------------
@torch.no_grad()
def embed_images(images, batch_size=4):
    # One (n_tokens, 128) float32 CPU tensor per page.
    out = []
    for i in tqdm(range(0, len(images), batch_size), desc="Embedding pages"):
        batch = processor.process_images(images[i:i + batch_size]).to(model.device)
        out.extend(model(**batch).float().cpu().unbind(0))
    return out


@torch.no_grad()
def embed_queries(queries, batch_size=16):
    # One (n_query_tokens, 128) float32 CPU tensor per query.
    out = []
    for i in range(0, len(queries), batch_size):
        batch = processor.process_queries(queries[i:i + batch_size]).to(model.device)
        out.extend(model(**batch).float().cpu().unbind(0))
    return out


# ---- Token merging (official hierarchical pooling) --------------------------
_pool_cache = {}   # (page id, factor) -> pooled tensor; probing factors and evaluating share poolings

def pool_page(embedding, factor):
    # Merge one page's patch embeddings into ~n/factor vectors (factor 1 = unchanged).
    key = (id(embedding), int(factor))
    if key not in _pool_cache:
        _pool_cache[key] = embedding if factor <= 1 else pooler.pool_embeddings([embedding], pool_factor=int(factor))[0]
    return _pool_cache[key]


# ---- Retrieval scoring -------------------------------------------------------
def maxsim_matrix(q_embs, d_embs):
    # Late-interaction score s(q, p) = sum over query tokens of max over page tokens (paper, Section 3).
    scores = torch.zeros(len(q_embs), len(d_embs))
    for j, d in enumerate(d_embs):
        for i, q in enumerate(q_embs):
            scores[i, j] = (q @ d.T).max(dim=1).values.sum()
    return scores


def retrieval_metrics(scores, relevant, k=K):
    # Per-query NDCG@k, Recall@1 and MRR for single-relevant-page queries.
    rows = torch.arange(len(relevant))
    rel_scores = scores[rows, torch.as_tensor(relevant)]
    rank = (1 + (scores > rel_scores[:, None]).sum(dim=1)).numpy()      # rank 1 = best
    return {
        f"ndcg@{k}": np.where(rank <= k, 1.0 / np.log2(rank + 1), 0.0),
        "recall@1": (rank == 1).astype(float),
        "mrr": 1.0 / rank,
    }


# ---- One benchmark = one corpus scored under every method -------------------
def run_benchmark(d_embs, q_embs, relevant, groups, S):
    # Scores Full / Static / Dynamic / memory-matched Static on one corpus.
    # d_embs: page embeddings; q_embs: query embeddings; relevant[i]: page index of query i;
    # groups[i]: dataset label of query i; S: density score per page.
    n_full = np.array([len(e) for e in d_embs])
    dyn_factors = np.array([get_thesis_factor(s) for s in S])
    groups = np.asarray(groups)

    def evaluate(factors):
        pooled = [pool_page(e, f) for e, f in tqdm(list(zip(d_embs, factors)), desc="Merging", leave=False)]
        return np.array([len(p) for p in pooled]), retrieval_metrics(maxsim_matrix(q_embs, pooled), relevant)

    methods = [
        ("Full (no merging)", "full", np.ones(len(d_embs), dtype=int)),
        (f"Static {STATIC_FACTOR} (paper)", "static", np.full(len(d_embs), STATIC_FACTOR)),
        ("Thesis Dynamic", "dynamic", dyn_factors),
    ]
    results = {}
    for name, role, factors in methods:
        tokens, metrics = evaluate(factors)
        results[name] = dict(role=role, factors=factors, tokens=tokens, metrics=metrics)

    ndcg_key = f"ndcg@{K}"
    full_ndcg = results["Full (no merging)"]["metrics"][ndcg_key].mean()
    summary, by_group = [], {}
    for name, r in results.items():
        m = r["metrics"]
        summary.append({
            "Method": name,
            "Effective factor": n_full.sum() / r["tokens"].sum(),
            "Tokens / page": r["tokens"].mean(),
            "Memory vs full (%)": 100 * r["tokens"].sum() / n_full.sum(),
            "Index size (KB)": r["tokens"].sum() * EMBED_DIM * BYTES_PER_VALUE / 1024,
            f"NDCG@{K}": m[ndcg_key].mean(),
            f"NDCG@{K} retained (%)": 100 * m[ndcg_key].mean() / full_ndcg,
            "Recall@1": m["recall@1"].mean(),
            "MRR": m["mrr"].mean(),
        })
        by_group[name] = pd.Series(m[ndcg_key]).groupby(groups).mean()

    page_groups = np.array(groups_per_page(groups, relevant, len(d_embs)))
    factor_by_group = pd.DataFrame({n: pd.Series(r["factors"], dtype=float).groupby(page_groups).mean()
                                    for n, r in results.items()})

    pages = pd.DataFrame({"group": page_groups, "S": S, "dynamic factor": dyn_factors,
                          "full tokens": n_full, "dynamic tokens": results["Thesis Dynamic"]["tokens"]})
    return {"summary": pd.DataFrame(summary).set_index("Method"),
            "by_group": pd.DataFrame(by_group),
            "factor_by_group": factor_by_group,
            "pages": pages,
            "roles": {n: r["role"] for n, r in results.items()}}


def groups_per_page(groups, relevant, n_pages):
    # Dataset label of each page (taken from the query that targets it).
    labels = [""] * n_pages
    for g, p in zip(groups, relevant):
        labels[p] = g
    return labels


# ---- Plot: retrieval accuracy per dataset (the trade-off lives in the frontier figure) ----
def plot_benchmark(res, title):
    by_group, roles, fbg = res["by_group"], res["roles"], res["factor_by_group"]
    fig, ax = plt.subplots(figsize=(11, 5.5))
    x = np.arange(len(by_group.index))
    # Exactly two lines: the paper's static baseline vs our dynamic method.
    # Static is drawn ON TOP with dashes so both lines stay visible where they overlap (ties).
    style = {"static": dict(linestyle="--", linewidth=2, marker="s", markersize=6.5, zorder=3),
             "dynamic": dict(linestyle="-", linewidth=3, marker="o", markersize=10, zorder=2)}
    plotted = sorted([n for n in by_group.columns if roles[n] in style],
                     key=lambda n: roles[n] != "dynamic")   # Thesis Dynamic first in the legend
    for name in plotted:
        role = roles[name]
        label = f"{name} (ours)" if role == "dynamic" else f"{name}: F={STATIC_FACTOR} on every page"
        ax.plot(x, by_group[name], label=label, color=ROLE_COLORS[role],
                markeredgecolor="white", markeredgewidth=1.2, **style[role])
        if role == "dynamic":   # mean merging factor Dynamic chose, at every point (static is always F=9)
            static_col = next(n for n in by_group.columns if roles[n] == "static")
            for xi, g in zip(x, by_group.index):
                v = by_group.loc[g, name]
                below = by_group.loc[g, static_col] > v   # static line sits above: put the label underneath
                ax.annotate(f"F={fbg.loc[g, name]:.0f}", (xi, v + (-0.016 if below else 0.016)),
                            ha="center", va="top" if below else "bottom", fontsize=9.5,
                            color=ROLE_COLORS[role], fontweight="bold",
                            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))
    ax.set_xticks(x)
    ax.set_xticklabels(by_group.index)
    ax.set_ylabel(f"NDCG@{K}")
    lo = by_group[plotted].values.min()
    ax.set_ylim(max(0, lo - 0.12), min(1.03, by_group[plotted].values.max() + 0.08))
    ax.grid(True, axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="lower right", fontsize=9)
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    fname = RESULTS_DIR / (title.split(":")[0].strip().lower().replace(" ", "_") + ".png")
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.show()

## 4. Benchmark A: ViDoRe corpus (50 real pages)
The statistical experiment. `evaluation_docs/` ships with this notebook and holds **54 real PDF documents** (50 single-page + 4 multi-page of 4-5 pages, 68 pages in total) drawn from five ViDoRe test splits: text-rich *DocVQA* and *TAT-DQA*, low-density *InfoVQA*, *ArxivQA* and *TabFQuAD*. `manifest.csv` is the ground truth: one row per (document, page) with the **real benchmark query whose answer is on that page**. Exactly as in production VDR, the pipeline renders each PDF to page images itself and feeds those to ColPali - the page, not the file, is the retrieval unit. Open the PDFs to see exactly which documents the numbers refer to.

All 50 pages are pooled into ONE retrieval corpus: when we ask query #17, the system must rank page #17 above the other 49 pages, which include both similar pages from the same dataset and very different ones. A merging method that destroys discriminative detail will visibly push correct pages down this ranking. (The old notebook scored one generic query against each page separately - that measures nothing, because there was nothing to retrieve *against*.)

In [ ]:
# Load the evaluation corpus: render every PDF document to page images (ColPali's indexing unit is the page).
# manifest.csv: one row per (document, page) with the real query whose answer is on that page.
manifest = pd.read_csv(EVAL_DIR / "manifest.csv")
rendered = {f: convert_from_path(EVAL_DIR / f, dpi=150)
            for f in tqdm(manifest["file"].unique(), desc="Rendering PDFs")}
pages_a = [
    {"doc": r.file, "dataset": r.dataset, "page_type": r.page_type, "query": r.query,
     "image": rendered[r.file][r.page].convert("RGB")}
    for r in manifest.itertuples()
]
n_docs = manifest["file"].nunique()
print(f"{n_docs} documents -> {len(pages_a)} pages loaded from {EVAL_DIR}/ "
      f"({(manifest.groupby('file').size() > 1).sum()} documents are multi-page)")
display(manifest.groupby(["dataset", "page_type"]).agg(documents=("file", "nunique"), pages=("file", "size")).reset_index())

In [ ]:
# Embed pages + queries once; checkpointed to cache/ so a crashed runtime resumes here instantly.
images_a = [p["image"] for p in pages_a]
d_embs_a = cached("vidore_pages", lambda: embed_images(images_a))
q_embs_a = cached("vidore_queries", lambda: embed_queries([p["query"] for p in pages_a]))
S_a = np.array([get_density_score(e) for e in d_embs_a])   # density measured on the embeddings themselves

res_a = run_benchmark(d_embs_a, q_embs_a, relevant=np.arange(len(pages_a)),
                      groups=[p["dataset"] for p in pages_a], S=S_a)

In [ ]:
# Results: summary table, per-dataset accuracy, density/factor assignment, plots.
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("Overall (54 documents, 68-page corpus)")
display(res_a["summary"])
print(f"\nNDCG@{K} per dataset")
display(res_a["by_group"])
print("\nDensity score S and the factor Thesis Dynamic assigned, per dataset")
display(res_a["pages"].groupby("group", sort=False).agg(S=("S", "mean"), dynamic_factor=("dynamic factor", "mean")))
plot_benchmark(res_a, "Benchmark A: 54 ViDoRe documents (68 pages)")

### 4b. The full accuracy-vs-memory frontier
Four points don't show the whole trade-off. Here we sweep **every static factor from 2 to 49** on the same embeddings to draw the complete static frontier, then place Thesis Dynamic against it. Dynamic adds value only if its point lies **above** the frontier at its memory budget; below the curve means a plain static factor of the same size would have done better.

In [ ]:
# Sweep static factors on the already-computed embeddings (no model calls - just merging + scoring).
n_full_a = np.array([len(e) for e in d_embs_a])
relevant_a = np.arange(len(pages_a))

def sweep_point(factors):
    pooled = [pool_page(e, f) for e, f in zip(d_embs_a, factors)]
    m = retrieval_metrics(maxsim_matrix(q_embs_a, pooled), relevant_a)
    return 100 * sum(len(p) for p in pooled) / n_full_a.sum(), m[f"ndcg@{K}"].mean()

_, full_ndcg_a = sweep_point(np.ones(len(d_embs_a), dtype=int))
frontier = []
for f in tqdm([2, 4, 6, 9, 13, 19, 25, 36, 49], desc="Static sweep"):
    mem, ndcg = sweep_point(np.full(len(d_embs_a), f))
    frontier.append({"kind": "static", "param": f, "memory_pct": mem, "ndcg5": ndcg,
                     "retained_pct": 100 * ndcg / full_ndcg_a})
for D in tqdm([6, 9, 13, 19, 25], desc="Dynamic sweep"):   # same analysis, default factor swept
    mem, ndcg = sweep_point(np.array([get_thesis_factor(s, default=D) for s in S_a]))
    frontier.append({"kind": "dynamic", "param": D, "memory_pct": mem, "ndcg5": ndcg,
                     "retained_pct": 100 * ndcg / full_ndcg_a})
frontier = pd.DataFrame(frontier)
frontier.to_csv(RESULTS_DIR / "static_frontier.csv", index=False)
stat, dyn = frontier[frontier.kind == "static"], frontier[frontier.kind == "dynamic"]
dyn_mem, dyn_ndcg = sweep_point(np.array([get_thesis_factor(s) for s in S_a]))

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.plot(dyn["memory_pct"], dyn["retained_pct"], "-o", color=ROLE_COLORS["dynamic"], linewidth=2.5,
        markersize=9, markeredgecolor="white", label="Thesis Dynamic (ours) - default factor D swept", zorder=4)
ax.plot(stat["memory_pct"], stat["retained_pct"], "--s", color=ROLE_COLORS["static"], linewidth=2,
        markersize=7, markeredgecolor="white", label="Static merging (paper) - one factor for all pages", zorder=3)
for _, r in stat.iterrows():
    ax.annotate(f"F={int(r['param'])}", (r["memory_pct"], r["retained_pct"]),
                textcoords="offset points", xytext=(6, -12), fontsize=8, color="#52514e")
for _, r in dyn.iterrows():
    ax.annotate(f"D={int(r['param'])}", (r["memory_pct"], r["retained_pct"]),
                textcoords="offset points", xytext=(4, 8), fontsize=8.5,
                color=ROLE_COLORS["dynamic"], fontweight="bold")
ax.scatter([100], [100], s=110, color=ROLE_COLORS["full"], edgecolor="white", linewidth=2, zorder=3, label="Full (no merging)")
ax.set_xscale("log")
ax.set_xticks([2, 3, 5, 10, 20, 50, 100])
ax.set_xticklabels(["2%", "3%", "5%", "10%", "20%", "50%", "100%"])
ax.set_xlabel("Index memory vs. full ColPali (log scale)")
ax.set_ylabel(f"NDCG@{K} retained vs. full (%)")
ax.set_title("Accuracy vs. memory: static frontier and the dynamic method")
ax.grid(True, alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right", fontsize=9)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "tradeoff_frontier.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Benchmark B: a multi-paper corpus (case study)
A corpus of **four research papers** (~59 pages): the target ACL paper plus three related papers (ColPali, Token Pooling, DSE) in `papers/` acting as distractors. Each query below is answered by one specific page of the **ACL paper** (0-based index, checked against the PDF) - but the system must now find that page among all pages of all four papers, which are deliberately similar in topic and look. A single 13-page paper alone is too easy: every method scores perfectly and nothing is learned. With only 8 queries this remains a qualitative case study.

In [ ]:
pdf_path = next((p for p in PDF_CANDIDATES if os.path.exists(p)), None)
if pdf_path is None:
    raise FileNotFoundError(f"Upload 2025.findings-acl.1003.pdf next to the notebook (looked in {PDF_CANDIDATES}).")

# Target paper FIRST (so the labelled page indices below stay 0-12), then the distractor papers.
pdf_pages, page_src = [], []
for path in [pdf_path] + sorted(str(p) for p in Path("papers").glob("*.pdf")):
    pages = convert_from_path(path, dpi=150)
    pdf_pages += pages
    page_src += [Path(path).stem] * len(pages)
    print(f"{path}: {len(pages)} pages")
print(f"Corpus B: {len(pdf_pages)} pages from {len(set(page_src))} papers")

# (query, 0-based index of the ACL-paper page that answers it)
PDF_QUERIES = [
    ("How does memory usage compare with NDCG@5 for ColPali, Light-ColPali and DSE?",           0),   # Figure 1
    ("Why does random token pruning outperform score-oriented and attention-oriented pruning?",   3),   # Section 4.2-4.3, Fig. 2-3
    ("Retrieval performance versus merging factor for 1D pooling, 2D pooling and clustering",     4),   # Figure 4
    ("Retrieval performance of pre-encoder, post-encoder, post-LLM and post-projector merging",   6),   # Table 1
    ("NDCG@5 of Light-ColPali and Light-ColQwen2 at merging factors 4, 9, 25 and 49",             7),   # Table 2
    ("Training and embedding generation time of ColQwen2 versus Light-ColQwen2 on A100 GPUs",     7),   # Table 3
    ("Prompt used to synthesize five questions from a document page",                             11),  # Listing 1
    ("NDCG@5 of retrievers built on Qwen2-VL-7B",                                                 12),  # Table 4
]

In [ ]:
d_embs_b = cached("acl_pages", lambda: embed_images(pdf_pages))
q_embs_b = cached("acl_queries", lambda: embed_queries([q for q, _ in PDF_QUERIES]))
S_b = np.array([get_density_score(e) for e in d_embs_b])

res_b = run_benchmark(d_embs_b, q_embs_b, relevant=[p for _, p in PDF_QUERIES],
                      groups=["ACL paper"] * len(PDF_QUERIES), S=S_b)
page_no = {}
labels = []
for s in page_src:
    page_no[s] = page_no.get(s, 0) + 1
    labels.append(f"{s} p.{page_no[s]}")
res_b["pages"]["group"] = labels

print(f"Overall ({len(pdf_pages)}-page corpus from {len(set(page_src))} papers, {len(PDF_QUERIES)} queries)")
display(res_b["summary"])
print("\nPer-page density and dynamic factor (target paper only)")
display(res_b["pages"].head(13).set_index("group")[["S", "dynamic factor", "full tokens", "dynamic tokens"]])
plot_benchmark(res_b, "Benchmark B: multi-paper corpus (4 papers)")

## 6. Final comparison: Thesis Dynamic vs. the paper's static baseline
One table for both benchmarks. The three questions it answers:

1. **Dynamic vs. Static 9:** accuracy *and* memory, since the two use different budgets.
2. **Dynamic vs. the static frontier (Section 4b):** does the dynamic point sit above the curve of what a plain static factor achieves at the same index size?
3. **Where does dynamic help or hurt?** The per-dataset NDCG@5 above (the paper reports that text-rich pages suffer most).

In [ ]:
cols = ["Effective factor", "Tokens / page", "Memory vs full (%)", f"NDCG@{K}", f"NDCG@{K} retained (%)", "Recall@1", "MRR"]
final = pd.concat({"A: ViDoRe (54 docs / 68 pages)": res_a["summary"][cols], "B: 4 papers (59 pages)": res_b["summary"][cols]},
                  names=["Benchmark"])
final.to_csv(RESULTS_DIR / "final_comparison.csv")
res_a["summary"].to_csv(RESULTS_DIR / "benchmark_A_summary.csv")
res_a["by_group"].to_csv(RESULTS_DIR / "benchmark_A_per_dataset.csv")
res_a["pages"].to_csv(RESULTS_DIR / "benchmark_A_pages.csv", index=False)
res_b["summary"].to_csv(RESULTS_DIR / "benchmark_B_summary.csv")
print(f"Tables saved to {RESULTS_DIR}/")
display(final)

for bench, res in {"A: ViDoRe": res_a, "B: ACL paper": res_b}.items():
    s = res["summary"]
    static, dyn = s.loc[f"Static {STATIC_FACTOR} (paper)"], s.loc["Thesis Dynamic"]
    print(f"\n{bench}")
    print(f"  Dynamic vs Static {STATIC_FACTOR}: NDCG@{K} {dyn[f'NDCG@{K}'] - static[f'NDCG@{K}']:+.3f}, "
          f"memory {dyn['Memory vs full (%)']:.1f}% vs {static['Memory vs full (%)']:.1f}% of full")

### 6b. Auto-generated write-up
A plain-language summary of where the dynamic method gains or loses accuracy and what it costs in memory, generated from the numbers above and saved to `results/RESULTS.md`.

In [ ]:
def write_summary(res_a, res_b):
    s, bg, fb = res_a["summary"], res_a["by_group"], res_a["factor_by_group"]
    FULL, STATIC, DYN = "Full (no merging)", f"Static {STATIC_FACTOR} (paper)", "Thesis Dynamic"
    nk = f"NDCG@{K}"
    L = ["# Results write-up: Dynamic vs. Static token merging", ""]
    L += [f"*Corpus A: {len(res_a['pages'])} pages from 5 ViDoRe datasets, one real query per page. "
          f"Metric: NDCG@5 (higher = correct page ranked nearer the top). Model: ColPali v1.2, training-free.*", ""]

    L += ["## Headline numbers (Benchmark A)", ""]
    L += ["| Method | Memory vs full | NDCG@5 | Retained |", "|---|---|---|---|"]
    for name in s.index:
        r = s.loc[name]
        L += [f"| {name} | {r['Memory vs full (%)']:.1f}% | {r[nk]:.3f} | {r[f'{nk} retained (%)']:.1f}% |"]
    dmem = s.loc[DYN, "Memory vs full (%)"] - s.loc[STATIC, "Memory vs full (%)"]
    dacc = s.loc[DYN, nk] - s.loc[STATIC, nk]
    L += ["", f"**Overall:** Dynamic vs Static {STATIC_FACTOR}: {dacc:+.3f} NDCG@5 "
          f"({'gain' if dacc > 0 else 'loss' if dacc < 0 else 'tie'}) at {dmem:+.1f} points of memory.", ""]
    kb_full, kb_stat, kb_dyn = (s.loc[n, "Index size (KB)"] for n in (FULL, STATIC, DYN))
    L += [f"**Total index for these {res_a['pages']['group'].size} pages:** full ColPali {kb_full/1024:.1f} MB -> "
          f"Dynamic {kb_dyn/1024:.2f} MB (a {100*(1-kb_dyn/kb_full):.0f}% memory saving) while retaining "
          f"{s.loc[DYN, f'{nk} retained (%)']:.1f}% of accuracy; Static {STATIC_FACTOR} needs {kb_stat/1024:.2f} MB "
          f"for {s.loc[STATIC, f'{nk} retained (%)']:.1f}%.", ""]

    L += ["## Where the accuracy moves (per dataset, Dynamic - Static 9)", ""]
    L += ["| Dataset | Static 9 | Dynamic | Delta | Mean F chosen | Reading |", "|---|---|---|---|---|---|"]
    for g in bg.index:
        d = bg.loc[g, DYN] - bg.loc[g, STATIC]
        f = fb.loc[g, DYN]
        verdict = ("gain" if d > 0.004 else "loss" if d < -0.004 else "tie")
        reading = {"gain": f"easing to F={f:.0f} recovered accuracy here",
                   "loss": f"F={f:.0f} on this data cost accuracy",
                   "tie": "merging choice made no measurable difference"}[verdict]
        L += [f"| {g} | {bg.loc[g, STATIC]:.3f} | {bg.loc[g, DYN]:.3f} | {d:+.3f} | {f:.1f} | {verdict}: {reading} |"]

    worst = (bg[DYN] - bg[FULL]).idxmin()
    L += ["", f"Hardest dataset for merging overall: **{worst}** "
          f"(Dynamic {bg.loc[worst, DYN]:.3f} vs Full {bg.loc[worst, FULL]:.3f}).", ""]

    sb = res_b["summary"]
    L += ["## Benchmark B (multi-paper corpus, 8 queries - qualitative only)", ""]
    L += [f"Dynamic: {sb.loc['Thesis Dynamic', nk]:.3f} at {sb.loc['Thesis Dynamic', 'Memory vs full (%)']:.1f}% memory; "
          f"Static {STATIC_FACTOR}: {sb.loc[STATIC, nk]:.3f} at {sb.loc[STATIC, 'Memory vs full (%)']:.1f}%.", ""]
    L += ["## Caveats", "",
          "- Training-free setting: the paper's 97.8% at factor 9 used fine-tuning; compare methods with each other.",
          "- With ~10 queries per dataset, differences under ~0.02 NDCG@5 are within noise.",
          "- Density thresholds were calibrated once on this corpus (see Section 3)."]
    text = "\n".join(L)
    (RESULTS_DIR / "RESULTS.md").write_text(text)
    print(text)

write_summary(res_a, res_b)